# Adaptive Pick-Your-Path Game with LangGraph

An AI-powered interactive storytelling engine that brings classic "pick-your-path" adventure games to life — where players type whatever they want, and an intelligent agent figures out what they actually meant

**Background**

What is a Pick your Path game?
A pick-your-path game (also known as a choose-your-own-adventure or interactive fiction game) is a story-driven game where the player shapes the narrative by making choices at key moments. Instead of following a single fixed plot, the player is presented with a scene, then chooses from a set of possible actions — each leading to a different scene, and ultimately to one of several possible endings. The result is a branching story tree, where the same starting point can lead to wildly different outcomes depending on the path the player takes.

![pick_your_path_story_tree.svg](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/-tzTdhAgbnBvDbS-_qNB2w/pick-your-path-story-tree.svg)

**Setup**
Installing Required Libraries

In [1]:
from typing import TypedDict, List, Dict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from openai import OpenAI

ModuleNotFoundError: No module named 'langchain_openai'

In [3]:
!pip install langchain_openai
from typing import TypedDict, List, Dict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from openai import OpenAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.5 MB/s eta 0:00:00


Configuring the LLM

In [5]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY" # Replace with your actual OpenAI API Key

# Initialize the OpenAI client
openai_client = OpenAI()

# Initialize the LangChain ChatOpenAI wrapper (used by LangGraph nodes)
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0.3)

print("LLM client initialized.")

LLM client initialized.


SCENES

In [ ]:

SCENES = {
    "wake_up": {
        "description": (
            "Your alarm goes off at 7:00 AM. Today is the day of your big group "
            "presentation in Professor Mendel's class — the one worth 40% of your grade. "
            "You have one hour before you need to leave. You're a little nervous, so "
            "maybe you should rehearse, but you're also hungry so you could make "
            "breakfast. Or maybe you should just go early and get it over with."
        ),
        "actions": {
            "rehearse": {
                "description": "Rehearse your part one more time in the mirror.",
                "next_scene": "rehearse_in_mirror"
            },
            "make_breakfast": {
                "description": "Make a real breakfast — eggs, toast, the works.",
                "next_scene": "kitchen_with_priya"
            },
            "skip_to_coffee": {
                "description": "Skip everything and grab coffee from the cart by the lecture hall.",
                "next_scene": "hallway_outside_hall"
            }
        }
    },

    "rehearse_in_mirror": {
        "description": (
            "You stand in front of the mirror, running through your section. "
            "Halfway through, you catch something — slide 4 says 'financial forecats' instead "
            "of 'forecasts.' Yikes."
        ),
        "actions": {
            "fix_typo": {
                "description": "Fix the typo and text your group so they know.",
                "next_scene": "enter_lecture_hall"
            },
            "leave_typo": {
                "description": "Leave it. Surely no one will notice.",
                "next_scene": "hallway_outside_hall"
            }
        }
    },

    "kitchen_with_priya": {
        "description": (
            "At the kitchen table, you bump into your roommate Priya — who happens to "
            "be in your group. She's pale. 'I think I'm coming down with something. "
            "Can you cover my slides if I tank?'"
        ),
        "actions": {
            "review_with_priya": {
                "description": "Yes and review her slides together.",
                "next_scene": "enter_lecture_hall"
            },
            "reassure_and_leave": {
                "description": "No, tell her she'll be fine, and head out early.",
                "next_scene": "hallway_outside_hall"
            }
        }
    },

    "hallway_outside_hall": {
        "description": (
            "You're standing in the hallway outside the lecture hall. You can hear "
            "the muffled voice of the group presenting before you. Your stomach flips. "
            "You need to calm down somehow."
        ),
        "actions": {
            "take_breaths": {
                "description": "Take three slow breaths and visualize the opening line.",
                "next_scene": "mid_presentation"
            },
            "skim_notes": {
                "description": "Pull out your phone to distract yourself..",
                "next_scene": "at_podium_shaking"
            }
        }
    },

    "enter_lecture_hall": {
        "description": (
            "You walk into the lecture hall feeling unusually prepared. The rest of your "
            "group is huddled near the podium and they look relieved to see you. "
            "Professor Mendel asks who wants to go first."
        ),
        "actions": {
            "volunteer_first": {
                "description": "Raise your hand. Get it over with.",
                "next_scene": "ending_standing_ovation"
            },
            "wait_turn": {
                "description": "Let another group volunteer — buy yourself a few minutes.",
                "next_scene": "mid_presentation"
            }
        }
    },

    "mid_presentation": {
        "description": (
            "You're three minutes into your section and it's going well. Then you "
            "glance up and see Professor Mendel frowning slightly at the back of the room. "
            "You can't tell if it's at you or at something else. Maybe you should "
            "improvise a joke to break the tension. Maybe it's safer to just keep going."
        ),
        "actions": {
            "tell_joke": {
                "description": "Throw in a small joke to break the tension.",
                "next_scene": "ending_standing_ovation"
            },
            "read_from_notes": {
                "description": "Lock in and read straight from your notes to be safe.",
                "next_scene": "ending_solid_bplus"
            }
        }
    },

    "at_podium_shaking": {
        "description": (
            "You step up to the podium. Your hands are shaking. Two sentences in, "
            "you lose your place. The room is very quiet."
        ),
        "actions": {
            "apologize_restart": {
                "description": "Smile, apologize, and start the slide over.",
                "next_scene": "ending_solid_bplus"
            },
            "plow_forward": {
                "description": "Plow forward and hope no one noticed.",
                "next_scene": "ending_learning_experience"
            }
        }
    },

    "ending_standing_ovation": {
        "description": (
            "ENDING — Standing Ovation. You nail it. Your group lights up, the class "
            "actually claps, and Professor Mendel nods the slow nod that means an A. "
            "Your groupmate Priya buys you a coffee on the way out. Roll credits."
        ),
        "actions": {}
    },

    "ending_solid_bplus": {
        "description": (
            "ENDING — Solid B+. It wasn't a triumph, but you made it through with "
            "your dignity intact. Professor Mendel writes 'clear, if cautious' on the "
            "feedback form. You learn that showing up prepared matters more than being a star."
        ),
        "actions": {}
    },

    "ending_learning_experience": {
        "description": (
            "ENDING — A Learning Experience. The presentation falls apart, slide 4 "
            "with a typo about forecats instead of forecasts gets a snicker from "
            "the back row, and you spend the rest of the class staring at the "
            "floor. But hey — next semester you'll be the one who rehearses in the"
            " mirror. Character development counts for something."
        ),
        "actions": {}
    }
}



The visualization of the example scene schema.

In [7]:
![scene_flow_tree.svg](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/wkQ4f8H-OdN9lpHGjYja2g/scene-flow-tree.svg)

/bin/bash: -c: line 1: syntax error near unexpected token `https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/wkQ4f8H-OdN9lpHGjYja2g/scene-flow-tree.svg'
/bin/bash: -c: line 1: `[scene_flow_tree.svg](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/wkQ4f8H-OdN9lpHGjYja2g/scene-flow-tree.svg)'


In [ ]:

class GameState(TypedDict):
    """State schema for the game.

    This TypedDict defines the data the game agent tracks as it moves through each scene in the game.
    """
    current_scene: str
    player_input: str
    interpretation: str
    game_over: bool

In [ ]:

def render_scene(state: GameState):
    """Render the current scene."""
    scene = SCENES[state["current_scene"]]

    print("\n" + "=" * 50)
    print(scene["description"])

    return state


In [ ]:
class IntentMatch(BaseModel):
    action_key: str # The matched key
    matched: bool # Flag for if a key was matched

In [ ]:
def interpret_action(state: GameState):
    """Interpret the player input to match one of the possible actions based on the current scene.
    If no scene matches, repeatedly ask for new input. Update the game state with the interpreted action key.
    """
    print("\nWhat do you do?\n")
    state["player_input"] = input("\n> ")

    scene = SCENES[state["current_scene"]]
    user_input = state["player_input"]
    possible_actions = {
        k: {
            "description": v["description"],
        }
        for k, v in scene["actions"].items()
    }

    structured = llm.with_structured_output(IntentMatch)

    while True:
        prompt = f"""
        You are an intent classifier for a text adventure game.

        Player input:
        {user_input}

        Available actions:
        {possible_actions}

        Rules:
            - If the input clearly relates to one of the available actions, set matched=True and return the best matching action_key.
            - If the input is cannot reasonably map to any available action, set matched=False.
            - When matched=False, action_key should be an empty string.
            - Return JSON only
        """

        result = structured.invoke(prompt)

        if result.matched and result.action_key in possible_actions:
            state["interpretation"] = result.action_key
            return state

        print("\nHmm, that doesn't quite fit. Try something else.")
        state["player_input"] = input("\n> ")
        user_input = state["player_input"]

In [ ]:
def transition(state: GameState):
    """Move from the current scene to the next scene as indicated by the intended action.
    If the action is invalid, do not move forward.
    """
    scene = SCENES[state["current_scene"]]
    action_key = state["interpretation"]

    # Safety Check
    if action_key not in scene["actions"]:
        print("You hesitate... unsure how to do that.")
        return state

    next_scene = scene["actions"][action_key]["next_scene"]
    state["current_scene"] = next_scene

    return state

In [ ]:
def check_end(state: GameState):
    """Check if the game is over."""

    if len(SCENES[state["current_scene"]]["actions"]) == 0:
        state["game_over"] = True

    return state

In [ ]:
workflow = StateGraph(GameState)

workflow.add_node("render_scene", render_scene)
workflow.add_node("interpret_action", interpret_action)
workflow.add_node("transition", transition)
workflow.add_node("check_end", check_end)

In [ ]:
workflow.add_edge("interpret_action", "transition")
workflow.add_edge("transition", "check_end")

In [ ]:
def route(state: GameState):
    """Route after render_scene.
    If the game is not over, proceed to get input.
    If the game is over, end the game.
    """
    if state["game_over"]:
        return "end_game" # A new key
    return "interpret_action"

workflow.add_conditional_edges(
    "render_scene",
    route, {
        "interpret_action": "interpret_action",
        "end_game": END # Map end_game key to END constant
    }
)

In [ ]:
workflow.add_edge(START, "render_scene")
workflow.add_edge("check_end", "render_scene")

In [ ]:
game = workflow.compile()

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(game.get_graph().draw_mermaid_png()))
except Exception:
    # If visualization dependencies aren't available, print text representation
    print("                   ┌─────────┐")
    print("                   │  START  │")
    print("                   └────┬────┘")
    print("                        │")
    print("                        ▼")
    print("           ┌───────────────────────┐")
    print("       ┌──▶│      render_scene     │")
    print("       │   └────────────┬──────────┘")
    print("       │                │")
    print("       │          (conditional)")
    print("       │          ╱            ╲")
    print("       │         ╱              ╲")
    print("       │        ▼                ▼")
    print("       │ ┌──────────────────┐  ┌───────┐")
    print("       │ │ interpret_action │  │  END  │")
    print("       │ └────────┬─────────┘  └───────┘")
    print("       │          │")
    print("       │          ▼")
    print("       │   ┌────────────┐")
    print("       │   │ transition │")
    print("       │   └─────┬──────┘")
    print("       │         │")
    print("       │         ▼")
    print("       │   ┌───────────┐")
    print("       │   │ check_end │")
    print("       │   └─────┬─────┘")
    print("       │         │")
    print("       └─────────┘")

In [ ]:
def run_game():
    # Define the initial state
    state = {
        "current_scene": #TODO: enter the first scene key ,
        "player_input": "",
        "interpretation": "",
        "game_over": False
    }

    # Run the workflow
    game.invoke(state)

    # When the workflow terminates:
    print("\nGame Over.")

run_game()

# Adaptive Pick-Your-Path Game with LangGraph

This project implements an AI-powered interactive storytelling engine using LangGraph, bringing classic "pick-your-path" adventure games to life. Players can type natural language inputs, and an intelligent agent interprets their intent to guide them through a branching narrative.

## Project Description

The game allows players to shape the narrative by making choices at key moments. Instead of a fixed plot, the player is presented with a scene and can choose from a set of possible actions. The AI, powered by OpenAI's language models, interprets these choices and transitions the player to different scenes, leading to various outcomes and endings.

## Technologies Used

*   **LangChain**: For building applications with large language models.
*   **LangGraph**: A library for building stateful, multi-actor applications with LLMs, used here to manage the game's state and transitions.
*   **OpenAI**: Provides the underlying large language model (`gpt-5.4-nano`) for interpreting player input.
*   **Python**: The core programming language.

## Setup and Installation

To set up and run this project, follow these steps:

1.  **Install Dependencies**: The project requires `langchain-openai` and `langgraph`. These are installed at the beginning of the notebook.

    ```python
    !pip install langchain_openai
    ```

2.  **OpenAI API Key**: You need an OpenAI API key to use the language model. Set your API key in the environment variable `OPENAI_API_KEY` within the notebook:

    ```python
    import os
    os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY" # Replace with your actual OpenAI API Key
    ```

    **Important**: Remember to replace `"YOUR_OPENAI_API_KEY"` with your actual OpenAI API key.

3.  **Run the Notebook Cells**: Execute all the cells in the notebook sequentially. This will:
    *   Define the game scenes (`SCENES`).
    *   Define the game state (`GameState`).
    *   Set up the `render_scene`, `interpret_action`, `transition`, and `check_end` functions.
    *   Construct the LangGraph workflow.
    *   Initialize and compile the game.

## How to Play

Once all the cells are executed, the `run_game()` function at the end of the notebook will start the game. You will be presented with the first scene, and then prompted to enter your actions. The game will interpret your input and guide you through the story until an ending is reached.

Example gameplay:

```
==================================================
Your alarm goes off at 7:00 AM. Today is the day of your big group presentation in Professor Mendel's class — the one worth 40% of your grade. You have one hour before you need to leave. You're a little nervous, so maybe you should rehearse, but you're also hungry so you could make breakfast. Or maybe you should just go early and get it over with.

What do you do?

> rehearse
==================================================
You stand in front of the mirror, running through your section. Halfway through, you catch something — slide 4 says 'financial forecats' instead of 'forecasts.' Yikes.

What do you do?

> fix the typo
==================================================
You walk into the lecture hall feeling unusually prepared. The rest of your group is huddled near the podium and they look relieved to see you. Professor Mendel asks who wants to go first.

What do you do?

> I'll go first
==================================================
ENDING — Standing Ovation. You nail it. Your group lights up, the class actually claps, and Professor Mendel nods the slow nod that means an A. Your groupmate Priya buys you a coffee on the way out. Roll credits.

Game Over.
```

Enjoy the adventure!